# Sit.3 — Entrenar ConvLSTM 2D + Positional Encoding

Predice NO2, SO2, O3 en T+1, T+3, T+7 a partir de secuencias
de 8 embeddings CLIP (generados por sit2_sae_posttrain.ipynb).

Arquitectura: X -> [+PE 25ch] -> ConvLSTM2D(64, k=3, 2capas) -> Conv1x1 -> AvgPool -> y=(3,3)
Loss: MSE enmascarada con pesos [10000, 1000, 1]
Dataset: Slucu-0310/geovision-cali-sit3



In [1]:
# @title Dependencias
%pip install -q huggingface_hub numpy pandas tqdm matplotlib
%pip install -q torch lightning

import warnings, os, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pl.seed_everything(SEED, workers=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")



Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.4/848.4 kB 27.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


Seed set to 42


Device: cuda


In [2]:
# @title Setup: crear modulos src/ localmente
import sys, os, shutil
sys.path.insert(0, ".")

# Limpiar modulos antiguos
for old_file in ["src/modelos/convlstm.py", "src/training/lit_convlstm.py"]:
    if os.path.exists(old_file):
        os.remove(old_file)
for cache_dir in ["src/__pycache__", "src/modelos/__pycache__", "src/training/__pycache__"]:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)

os.makedirs("src/modelos", exist_ok=True)
os.makedirs("src/training", exist_ok=True)

# ===== convlstm2d.py =====
with open("src/modelos/convlstm2d.py", "w") as f:
    f.write("""\
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvLSTMCell(nn.Module):
    'Celda ConvLSTM 2D individual.'
    def __init__(self, input_dim, hidden_dim, kernel_size=3, bias=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                               kernel_size, padding=padding, bias=bias)

    def forward(self, x, prev_state):
        h_prev, c_prev = prev_state
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.split(gates, self.hidden_dim, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c


class ConvLSTM2D(nn.Module):
    'ConvLSTM 2D + Positional Encoding aprendible (25ch).'

    def __init__(self, input_channels=522, pos_channels=25, hidden_dim=64,
                 kernel_size=3, num_layers=2, num_horizons=3, num_pollutants=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_horizons = num_horizons
        self.num_pollutants = num_pollutants
        self.pos_encoding = nn.Parameter(torch.randn(1, 1, pos_channels, 5, 5) * 0.1)
        total_input = input_channels + pos_channels
        self.cells = nn.ModuleList()
        for _ in range(num_layers):
            self.cells.append(ConvLSTMCell(total_input, hidden_dim, kernel_size))
            total_input = hidden_dim
        self.output_conv = nn.Conv2d(hidden_dim, num_horizons * num_pollutants, 1)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if "conv" in name and "weight" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

    def forward(self, x):
        'Forward pass con positional encoding y ConvLSTM.'
        N, T, C, H, W = x.shape
        pe = self.pos_encoding.expand(N, -1, -1, -1, -1).expand(-1, T, -1, -1, -1)
        x_pe = torch.cat([x, pe], dim=2)
        h = [torch.zeros(N, self.hidden_dim, H, W, device=x.device) for _ in range(self.num_layers)]
        c = [torch.zeros(N, self.hidden_dim, H, W, device=x.device) for _ in range(self.num_layers)]
        for t in range(T):
            inp = x_pe[:, t]
            for layer in range(self.num_layers):
                h[layer], c[layer] = self.cells[layer](inp, (h[layer], c[layer]))
                inp = h[layer]
        y_grid = self.output_conv(h[-1])
        return y_grid.mean(dim=[-1, -2]).view(N, self.num_horizons, self.num_pollutants)


def masked_mse_loss(y_pred, y_true, weights=None):
    'MSE loss que ignora valores NaN en y_true.'
    if weights is None:
        weights = torch.tensor([10000.0, 1000.0, 1.0], device=y_pred.device)
    weights = weights.to(y_pred.device)
    mask = ~torch.isnan(y_true)
    loss_p = torch.zeros(3, device=y_pred.device)
    for p in range(3):
        pmask = mask[:, :, p]
        if pmask.sum() > 0:
            loss_p[p] = F.mse_loss(y_pred[:, :, p][pmask], y_true[:, :, p][pmask], reduction="mean")
    return (loss_p * weights).sum() / weights.sum()""")
print("  Creado: src/modelos/convlstm2d.py (82 lines)")

# ===== lit_convlstm2d.py =====
with open("src/training/lit_convlstm2d.py", "w") as f:
    f.write("""\
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl
from src.modelos.convlstm2d import ConvLSTM2D, masked_mse_loss


class Sit3ConvLSTMDataset(Dataset):
    'Dataset que carga X e y desde archivos .npy.'
    def __init__(self, x_path, y_path, split_indices=None):
        self.X = np.load(x_path, mmap_mode="r")
        self.y = np.load(y_path, mmap_mode="r")
        if split_indices is not None:
            self.X = self.X[split_indices]
            self.y = self.y[split_indices]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx].copy()).float()
        y = torch.from_numpy(self.y[idx].copy()).float()
        return {"x": x, "y": y}


POLLUTANTS = ["NO2", "SO2", "O3"]
HORIZONS = ["T+1", "T+3", "T+7"]
CONV_FACTOR = {"NO2": 5750.0, "SO2": 8008.0, "O3": 6000.0}
KPI_UGM3 = {"NO2": 8.0, "SO2": 6.0, "O3": 12.0}


class LitConvLSTM2D(pl.LightningModule):
    'LightningModule para ConvLSTM2D (Situacion 3).'
    WEIGHTS_DEFAULT = [10000.0, 1000.0, 1.0]

    def __init__(self, model, lr=1e-4, weight_decay=1e-5, loss_weights=None,
                 y_mean=None, y_std=None):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.loss_weights = loss_weights or self.WEIGHTS_DEFAULT
        self.y_mean = y_mean
        self.y_std = y_std
        self._test_preds = []
        self._test_trues = []

    def forward(self, x):
        return self.model(x)

    def _loss(self, y_pred, y_true):
        w = torch.tensor(self.loss_weights, device=y_pred.device)
        return masked_mse_loss(y_pred, y_true, weights=w)

    def training_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        loss = self._loss(y_pred, batch["y"])
        self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        loss = self._loss(y_pred, batch["y"])
        self.log("val/loss", loss, on_epoch=True, prog_bar=True)
        self.log("val/rmse", loss.sqrt(), on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        y_true = batch["y"]
        self.log("test/loss", self._loss(y_pred, y_true), on_epoch=True)
        self._test_preds.append(y_pred.detach().cpu())
        self._test_trues.append(y_true.detach().cpu())

    def on_test_end(self):
        all_preds = torch.cat(self._test_preds, dim=0).numpy()
        all_trues = torch.cat(self._test_trues, dim=0).numpy()
        # Denormalizar si stats disponibles
        if self.y_mean is not None and self.y_std is not None:
            ym = np.array(self.y_mean).reshape(1, 1, 3)
            ys = np.array(self.y_std).reshape(1, 1, 3)
            all_preds = all_preds * ys + ym
            all_trues = all_trues * ys + ym
        mask = ~np.isnan(all_trues)
        print()
        print("=" * 70)
        print("  TABLA RESUMEN - CONVLSTM (SIT. 3)")
        print("=" * 70)
        loss_val = float(masked_mse_loss(torch.from_numpy(all_preds), torch.from_numpy(all_trues)))
        rmse_val = float(np.sqrt(loss_val))
        print(f"  Loss global (MSE):       {loss_val:.6e}")
        print(f"  RMSE global (mol/m2):    {rmse_val:.6f}")
        print()
        pnames = ["NO2", "SO2", "O3"]
        hnames = ["T+1", "T+3", "T+7"]
        cfac = {"NO2": 5750.0, "SO2": 8008.0, "O3": 6000.0}
        kpi_u = {"NO2": 8.0, "SO2": 6.0, "O3": 12.0}
        print("  Contaminante   RMSE mol/m2     RMSE ug/m3    KPI ug/m3  Cumple     R2")
        print("  " + "-"*15 + " " + "-"*14 + " " + "-"*12 + " " + "-"*10 + " " + "-"*8 + " " + "-"*8)
        for pi, pn in enumerate(pnames):
            pmask = mask[:, :, pi]
            y_p = all_preds[:, :, pi][pmask]
            y_t = all_trues[:, :, pi][pmask]
            if len(y_p) > 0:
                rp = float(np.sqrt(np.mean((y_p - y_t) ** 2)))
                ug = rp * cfac.get(pn, 1.0)
                ss_res = float(np.sum((y_p - y_t) ** 2))
                ss_tot = float(np.sum((y_t - np.mean(y_t)) ** 2))
                r2 = 1.0 - ss_res / (ss_tot + 1e-12) if ss_tot > 0 else float("nan")
                kp = kpi_u.get(pn, float("inf"))
                cp = "SI" if ug <= kp else "NO"
            else:
                rp, ug, r2, cp = float("nan"), float("nan"), float("nan"), "N/A"
            print(f"  {pn:15s} {rp:>8.2e}     {ug:>7.2f} ug/m3  {kp:>6.1f}     {cp:>8s}  {r2:>7.4f}")
        print()
        print("  Horizonte    RMSE mol/m2     R2")
        print("  " + "-"*10 + " " + "-"*14 + " " + "-"*8)
        for hi, hn in enumerate(hnames):
            hmask = mask[:, hi, :]
            y_ph = all_preds[:, hi, :][hmask]
            y_th = all_trues[:, hi, :][hmask]
            if len(y_ph) > 0:
                rh = float(np.sqrt(np.mean((y_ph - y_th) ** 2)))
                ss_res = float(np.sum((y_ph - y_th) ** 2))
                ss_tot = float(np.sum((y_th - np.mean(y_th)) ** 2))
                r2_h = 1.0 - ss_res / (ss_tot + 1e-12) if ss_tot > 0 else float("nan")
            else:
                rh, r2_h = float("nan"), float("nan")
            print(f"  {hn:10s} {rh:>8.2e}     {r2_h:>7.4f}")
        print()
        m1 = mask[:, 0, :]
        m7 = mask[:, 2, :]
        if m1.sum() > 0 and m7.sum() > 0:
            r1 = float(np.sqrt(np.mean((all_preds[:, 0, :][m1] - all_trues[:, 0, :][m1]) ** 2)))
            r7 = float(np.sqrt(np.mean((all_preds[:, 2, :][m7] - all_trues[:, 2, :][m7]) ** 2)))
            degradacion = (r7 / r1 - 1) * 100 if r1 > 0 else float("nan")
            cumple_d = "SI" if degradacion < 60 else "NO"
            print(f"  Degradacion T+1 -> T+7: {degradacion:.1f}%  (KPI < 60% -> {cumple_d})")
        else:
            print("  Degradacion T+1 -> T+7: N/A")
        print()
        print("=" * 70)
        self._test_preds.clear()
        self._test_trues.clear()

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10, min_lr=1e-7)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "val/loss"}}""")
print("  Creado: src/training/lit_convlstm2d.py (144 lines)")

print("Modulos src/ creados localmente")
print("OK: ConvLSTM2D con input_channels=522, sin duplicados")
print("OK: LitConvLSTM2D con tabla de metricas")


  Creado: src/modelos/convlstm2d.py (82 lines)
  Creado: src/training/lit_convlstm2d.py (144 lines)
Modulos src/ creados localmente
OK: ConvLSTM2D con input_channels=522, sin duplicados
OK: LitConvLSTM2D con tabla de metricas


## 1. Descargar dataset de HuggingFace

Dataset: Slucu-0310/geovision-cali-sit3 (X_convlstm.npy + y_convlstm.npy)



In [3]:
# @title Descargar dataset
from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

DATA_DIR = Path("/content/dataset_sit3")
if not (DATA_DIR / "X_convlstm.npy").is_file():
    print("Descargando dataset...")
    snapshot_download(
        repo_id="Slucu-0310/geovision-cali-sit3",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        token=HF_TOKEN,
    )
print("Dataset listo")



Descargando dataset...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Dataset listo


## 2. Cargar datos y crear splits

Se cargan X e y desde .npy y se dividen en 70/15/15
usando los splits del dataset original (heredados de Sit2).



In [4]:
# @title Cargar X, y y crear splits
X = np.load(DATA_DIR / "X_convlstm.npy", mmap_mode="r")
y = np.load(DATA_DIR / "y_convlstm.npy", mmap_mode="r")
print(f"X: {X.shape}  y: {y.shape}")

# Cargar metadatos para splits
import tempfile, os
os.environ["HF_TOKEN"] = HF_TOKEN or ""
import huggingface_hub
meta_path = huggingface_hub.hf_hub_download(
    repo_id="Slucu-0310/geovision-cali-sit2",
    repo_type="dataset",
    filename="metadatos.parquet",
    token=HF_TOKEN,
)
df_meta = pd.read_parquet(meta_path)

# Asignar split a cada secuencia
# Cada secuencia viene de una celda 0.05 con 8 tiles.
# Usamos los splits del primer tile de cada secuencia.
df_meta["celda_lat"] = (df_meta["centroide_lat"] / 0.05).round() * 0.05
df_meta["celda_lon"] = (df_meta["centroide_lon"] / 0.05).round() * 0.05

# Normalizar y con z-score (por contaminante)
y_data = y[:]  # load full array
y_mean = np.nanmean(y_data, axis=(0, 1), keepdims=True)  # (1,1,3) mean por contaminante
y_std = np.nanstd(y_data, axis=(0, 1), keepdims=True)    # (1,1,3) std por contaminante
y_std[y_std < 1e-12] = 1.0  # evitar division por cero
y_norm = (y_data - y_mean) / y_std
y_norm = np.where(np.isnan(y_data), np.nan, y_norm)  # preservar NaNs
print(f"Normalizacion z-score: mean={np.squeeze(y_mean)}, std={np.squeeze(y_std)}")

# Sobrescribir y con valores normalizados
y = y_norm.reshape(y.shape)

# Split aleatorio
from sklearn.model_selection import train_test_split

N = len(X)
indices = np.arange(N)
train_idx, tmp_idx = train_test_split(
    indices, test_size=0.3, random_state=SEED
)
val_idx, test_idx = train_test_split(
    tmp_idx, test_size=0.5, random_state=SEED
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")



X: (1403, 8, 522, 5, 5)  y: (1403, 3, 3)


metadatos.parquet:   0%|          | 0.00/335k [00:00<?, ?B/s]

Normalizacion z-score: mean=[2.9344143e-05 2.6640107e-04 1.1592717e-01], std=[1.7674871e-05 2.6904233e-04 5.4461695e-03]
Train: 982, Val: 210, Test: 211


In [5]:
# @title Crear datasets y dataloaders
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

BATCH_SIZE = 32



## 3. Crear modelo y entrenar

ConvLSTM 2D + Positional Encoding (25ch) -> ConvLSTM2D(64, k=3, 2 capas) -> Conv1x1 -> AvgPool -> 9 salidas



In [6]:
# @title Crear modelo
import sys
sys.path.insert(0, ".")
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.modelos.convlstm2d import ConvLSTM2D, masked_mse_loss
from src.training.lit_convlstm2d import Sit3ConvLSTMDataset

# Detectar canales reales del tensor
actual_channels = int(np.load(DATA_DIR / "X_convlstm.npy", mmap_mode="r").shape[2])
print(f"Tensor X canales detectados: {actual_channels}")

# Crear modelo con los canales reales
model = ConvLSTM2D(input_channels=actual_channels, hidden_dim=128, kernel_size=3, num_layers=3)
total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo ConvLSTM2D creado. Parametros: {total_params:,}")

# Guardar stats de normalizacion para LitConvLSTM2D
Y_MEAN = y_mean
Y_STD = y_std

# Crear datasets y dataloaders
ds_train = Sit3ConvLSTMDataset(str(DATA_DIR / "X_convlstm.npy"), str(DATA_DIR / "y_convlstm.npy"), split_indices=train_idx.tolist())
ds_val   = Sit3ConvLSTMDataset(str(DATA_DIR / "X_convlstm.npy"), str(DATA_DIR / "y_convlstm.npy"), split_indices=val_idx.tolist())
ds_test  = Sit3ConvLSTMDataset(str(DATA_DIR / "X_convlstm.npy"), str(DATA_DIR / "y_convlstm.npy"), split_indices=test_idx.tolist())

BATCH_SIZE = 32
dl_train = DataLoader(ds_train, BATCH_SIZE, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val,   BATCH_SIZE, shuffle=False, num_workers=0)
dl_test  = DataLoader(ds_test,  BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Batches: train={len(dl_train)}, val={len(dl_val)}, test={len(dl_test)}")


Tensor X canales detectados: 522
Modelo ConvLSTM2D creado. Parametros: 5,473,018
Batches: train=31, val=7, test=7


In [7]:
# @title Entrenar
from pathlib import Path
from lightning.pytorch.loggers import CSVLogger
import lightning.pytorch as pl

from src.training.lit_convlstm2d import LitConvLSTM2D
from src.modelos.convlstm2d import masked_mse_loss

RUN_DIR = Path("/content/runs/sit3_convlstm")
RUN_DIR.mkdir(parents=True, exist_ok=True)

lit_model = LitConvLSTM2D(model, lr=5e-5, weight_decay=1e-5,
                                  y_mean=Y_MEAN, y_std=Y_STD)

callbacks = [
    pl.callbacks.EarlyStopping(monitor="val/loss", patience=40, mode="min"),
    pl.callbacks.ModelCheckpoint(
        dirpath=str(RUN_DIR), filename="best",
        monitor="val/loss", mode="min", save_top_k=1
    ),
]

trainer = pl.Trainer(
    max_epochs=400, accelerator="auto", devices=1,
    callbacks=callbacks,
    logger=CSVLogger(save_dir=str(RUN_DIR / "lightning_logs")),
    log_every_n_steps=5, gradient_clip_val=1.0,
    default_root_dir=str(RUN_DIR),
)

trainer.fit(lit_model, dl_train, dl_val)
print(f"Entrenamiento completo. Mejor checkpoint en {RUN_DIR}")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ConvLSTM2D │  5.5 M │ train │     0 │
└───┴───────┴────────────┴────────┴───────┴───────┘

Trainable params: 5.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.5 M                                                                                                
Total estimated model params size (MB): 21.892                                                                     
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Entrenamiento completo. Mejor checkpoint en /content/runs/sit3_convlstm


## 4. Evaluacion en test



In [10]:
# @title Evaluar en test
import os, glob as gl

# Buscar el mejor checkpoint
ckpts = sorted(RUN_DIR.glob("best*"), key=os.path.getmtime, reverse=True)
if ckpts:
    ckpt_path = str(ckpts[0])
    print(f"Usando checkpoint: {ckpt_path}")
    trainer.test(lit_model, dl_test, ckpt_path=ckpt_path, weights_only=False)
else:
    print("Checkpoint no encontrado, evaluando con ultimo modelo")
    trainer.test(lit_model, dl_test, weights_only=False)

# --- Curvas de entrenamiento ---
csv_files = gl.glob(str(RUN_DIR / "lightning_logs" / "version_*" / "metrics.csv"))
if csv_files:
    import pandas as pd
    import matplotlib.pyplot as plt

    df_log = pd.read_csv(csv_files[0])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    if "train/loss_epoch" in df_log.columns:
        axes[0].plot(df_log["train/loss_epoch"], label="train", alpha=0.7)
    if "val/loss" in df_log.columns:
        axes[0].plot(df_log["val/loss"], label="val", alpha=0.7)
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("Loss de entrenamiento")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # RMSE
    if "val/rmse" in df_log.columns:
        axes[1].plot(df_log["val/rmse"], label="val RMSE", alpha=0.7)
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("RMSE global (val)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(str(RUN_DIR / "training_curves.png"), dpi=150)
    plt.show()
    print(f"Curvas guardadas en {RUN_DIR / 'training_curves.png'}")
else:
    print("No se encontraron logs de entrenamiento.")


Restoring states from the checkpoint path at /content/runs/sit3_convlstm/best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /content/runs/sit3_convlstm/best.ckpt


Output()

Usando checkpoint: /content/runs/sit3_convlstm/best.ckpt


======================================================================

TABLA RESUMEN - CONVLSTM (SIT. 3)

...

T+7        1.59e-05      1.0000

Degradacion T+1 -> T+7: 1.1%  (KPI < 60% -> SI)

======================================================================

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/loss         │   9.371142084546591e-09   │
└───────────────────────────┴───────────────────────────┘

No se encontraron logs de entrenamiento.


## 5. Curvas de entrenamiento



In [11]:
# @title Graficar loss y RMSE
# Leer metricas finales del CSV de test (epoch-level aggregation)
import glob as gl
test_csv = gl.glob(str(RUN_DIR / "lightning_logs" / "version_*" / "metrics.csv"))
if test_csv:
    df_log_all = pd.read_csv(test_csv[0])
    # Filter only test metrics (last rows)
    test_cols = [c for c in df_log_all.columns if c.startswith("test/") and c != "step"]
    if test_cols:
        last_row = df_log_all[test_cols].dropna(how="all").iloc[-1]
        print("\nMetricas finales (test):")
        for col in test_cols:
            val = last_row[col]
            if pd.notna(val):
                print(f"  {col}: {float(val):.6f}")

# Cargar historial desde lightning_logs
csv_files = gl.glob(str(RUN_DIR / "lightning_logs" / "version_*" / "metrics.csv"))
if csv_files:
    df_log = pd.read_csv(csv_files[0])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    if "train/loss_epoch" in df_log.columns:
        axes[0].plot(df_log["train/loss_epoch"], label="train", alpha=0.7)
    if "val/loss" in df_log.columns:
        axes[0].plot(df_log["val/loss"], label="val", alpha=0.7)
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("Loss de entrenamiento")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # RMSE
    for p in ["NO2", "SO2", "O3"]:
        col = "val/rmse/" + p
        if col in df_log.columns:
            axes[1].plot(df_log[col], label=p, alpha=0.7)
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("RMSE por contaminante (val)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(str(RUN_DIR / "training_curves.png"), dpi=150)
    plt.show()
    print(f"Curvas guardadas en {RUN_DIR / 'training_curves.png'}")
else:
    print("No se encontraron logs de entrenamiento.")



No se encontraron logs de entrenamiento.


## 6. Resumen de metricas

| KPI | Minimo | Medicion |
|-----|--------|----------|
| RMSE LOO-CV NO2 (T+1) | <= 8 ug/m3 | (test) |
| RMSE LOO-CV SO2 (T+1) | <= 6 ug/m3 | (test) |
| RMSE LOO-CV O3 (T+1) | <= 12 ug/m3 | (test) |
| R2 LOO-CV (promedio) | >= 0.55 | (test, targets absolutos) |
| Degradacion T+1->T+7 | < 60% | (test) |
